In [ ]:
import pandas as pd
import os
import re



Make connection to db

In [ ]:
import os
import re
import pandas as pd
from psycopg.pq import ExecStatus
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()

# FORCE the explicit synchronous psycopg2 driver
db_url = re.sub(r'^postgresql:', 'postgresql+psycopg2:', os.getenv('DATABASE_URL', ''))

engine = create_engine(
    db_url,
    pool_recycle=300,  # Cycles out stale connections
    pool_pre_ping=True # Checks live connection health BEFORE trying execution
)

# Pandas block
orders_test = pd.read_sql("SELECT * FROM orders LIMIT 5;", con=engine)
print(orders_test)


read db and get last watermark date

In [ ]:
last_updated_query = "select max(updated_at) from orders"
last_updated_run = pd.read_sql(last_updated_query,con=engine)
last_update_time = pd.to_datetime(last_updated_run.iloc[0,0]).tz_localize(None).strftime('%Y-%m-%d %H:%M:%S')
last_update_time

In [ ]:
date_to_load_from = last_update_time
df = pd.read_csv("orders_20261025.csv")


Clean data from csv creating the orders fact table
deduplicate, flag and the invalid orders, and lowercase the order status

In [ ]:
# get the rows I want for the fact table "Orders"
orders = df[["order_id","customer_id", "order_date", "product", "total_amount", "status","country", "updated_at"]]
#make sure the rows are from after the cutoffs
cutoff_time = pd.to_datetime(date_to_load_from)
orders["updated_at"] = pd.to_datetime(orders["updated_at"])
orders = orders[orders["updated_at"] >= cutoff_time].copy()
#drop duplicated and keep the last entry
orders = orders.drop_duplicates(subset="order_id", keep="last")
flagged_orders = orders[orders["total_amount"].isna()].copy()
#clean date columns to be a single format
orders['order_date'] = pd.to_datetime(orders['order_date'], format="mixed")
#upper country column
orders['country'] = orders['country'].str.upper()

#valid orders are te ones with a valid total amount for order... can change based on business needs
valid_orders = orders[orders["total_amount"].notna()].copy()
#lowercase the status' to make them more uniform for querying later on
valid_orders['status'] = valid_orders['status'].str.lower()
#clean total_amount by stripping the $ 
valid_orders['total_amount'] = valid_orders['total_amount'].str.replace('$','',regex=False).astype(float)
valid_orders


write flagged orders to a file for readability

In [ ]:
from datetime import datetime

if not flagged_orders.empty:
    run_date = datetime.now().strftime("%Y%m%d")
    flagged_orders.to_csv(f"flagged_orders_{run_date}.csv", index=False)

df for cutomer dimesion table

In [ ]:
customers = df[["customer_id", "customer_name", "email","country"]]
#clean customer name column
customers["customer_id"] = customers["customer_id"].astype(int)
customers['customer_name'] = customers['customer_name'].str.title()
customers['customer_name'] = customers['customer_name'].fillna("Missing")
customers = customers.drop_duplicates(subset=["customer_id"], keep="first")
customers = customers.drop_duplicates(subset=["customer_id"], keep="first")
#clean country column
customers['country'] = customers['country'].str.upper()
customers['country'] = customers['country'].fillna("Missing")
#clean email column
customers['email'] = customers['email'].fillna("Missing")
# customers.groupby("customer_id")[["country","customer_name", "email"]].agg(list)
len(customers)

46

dimesion table for products

In [ ]:
products = df[["product","category","unit_price"]]
products["unit_price"] = products["unit_price"].str.replace('$','').astype(float)
products['category'] = products['category'].str.title()
products['product'] = products['product'].str.title()
products = products.drop_duplicates(subset="product")

products

make the database connection

put dataframe data in database and count the rows in staging orders table and count rows in df and then perform a merge automatically if counts match up... if not alert for further inspection

In [ ]:
import pandas as pd
from sqlalchemy import text

valid_orders.to_sql("orders_temp", con=engine, if_exists="replace", index=False)
query = "SELECT count(order_id) from orders_temp"
orders_count = pd.read_sql(query, con=engine)
local_count = len(valid_orders)

# FIX: Added [0, 0] to extract the primitive integer value from the dataframe
db_count = int(orders_count.iloc[0, 0]) 
print(db_count)

if local_count == db_count:
    merge_query = text("""
    MERGE INTO orders AS target
    USING orders_temp AS source
    ON target.order_id = source.order_id::text
    WHEN MATCHED THEN
        UPDATE SET 
            total_amount = source.total_amount::double precision, 
            status = source.status, 
            updated_at = source.updated_at::timestamp,
            order_date = source.order_date::date,
            product = source.product,
            country =  source.country
    WHEN NOT MATCHED THEN
        INSERT (order_id, customer_id, order_date, product,  status, country, total_amount, updated_at)
        VALUES (
            source.order_id::text, 
            source.customer_id::integer,
            source.order_date::date,
            source.product,
            source.status, 
            source.country,
            source.total_amount::double precision, 
            source.updated_at::timestamp
        );
""") 
else:
    from datetime import datetime
    with open("error_log.txt", "a", encoding="utf-8") as error_log:
        error_log.write(str(f"Failed to execute merge query for products on {datetime.now()} because local products count and sql table counts were off"))
    products_merge = None
    merge_query = None

if merge_query is not None:
    with engine.begin() as connection:
        connection.execute(merge_query)
        connection.execute(text("drop table orders_temp;"))


Insert Products into a temp table

In [ ]:
products.to_sql(name='products_temp',
    con=engine,
    if_exists='replace',
    index=False
    )

merge to prod products table

In [ ]:
from datetime import datetime
sql_products_temp_count_query = "SELECT count(product) from products_temp"
products_df = pd.read_sql(sql_products_temp_count_query, con=engine)
local_products_count = len(products)
db_products_count = int(products_df.iloc[0,0])
print(local_products_count)
print(db_products_count)

if local_products_count == db_products_count:
    products_merge = text("""
        Merge into products as target
        Using products_temp as source 
        On target.product = source.product
        When Matched Then 
        Update Set
            category = source.category,
            unit_price = source.unit_price
        WHEN NOT MATCHED THEN
        Insert(product, category, unit_price)
        VALUES (source.product, source.category, source.unit_price)
        """
    )
else:
    with open("error_log.txt", "a") as error_log:
        error_log.write(str(f"Failed to execute merge query for products on {datetime.now} because local products count and sql table counts were off"))
    print("Mismatch with product counts")
    products_merge = None

if products_merge is not None:
    with engine.begin() as connection:
        connection.execute(products_merge)
        connection.execute(text("DROP table products_temp"))


Make a customers_temp table and then check the count of the rows and if the row counts match the perform a merge into the products table

In [107]:
import pandas as pd
from sqlalchemy import text

customers.to_sql("customers_temp", con=engine, if_exists="replace", index=False)
query = "SELECT count(customer_id) from customers_temp"
customers_count = pd.read_sql(query, con=engine)
local_count = len(customers)

# FIX: Added [0, 0] to extract the primitive integer value from the dataframe
db_count = int(customers_count.iloc[0, 0]) 
print(db_count)

if local_count == db_count:
    merge_query = text("""
    MERGE INTO customers AS target
    USING customers_temp AS source
    ON target.customer_id = source.customer_id::integer
    WHEN MATCHED THEN
        UPDATE SET
            customer_name = CASE WHEN source.customer_name = 'Missing' THEN target.customer_name ELSE source.customer_name END,
            email         = CASE WHEN source.email         = 'Missing' THEN target.email         ELSE source.email         END,
            country       = CASE WHEN source.country       = 'Missing' THEN target.country       ELSE source.country       END
    WHEN NOT MATCHED THEN
        INSERT (customer_id, customer_name, email, country)
        VALUES (source.customer_id::integer, source.customer_name, source.email, source.country);
""")
else:
    from datetime import datetime
    with open("error_log.txt", "a", encoding="utf-8") as error_log:
        error_log.write(str(f"Failed to execute merge query for customers on {datetime.now()} because local products count and sql table counts were off"))
    products_merge = None
    merge_query = None

if merge_query is not None:
    with engine.begin() as connection:
        connection.execute(merge_query)
        connection.execute(text("drop table customers_temp;"))


46
